# Detecting Attacks

In [1]:
import networkx as nx
from networkx.algorithms import community
import pandas as pd
import warnings
import os
os.chdir("..")
from util.load_graph import load_msg_graph, load_user_graphs, get_driver

warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
import pandas as pd

def extract_messages_from_graph(G):
    rows = []

    for node_id, attrs in G.nodes(data=True):

        if "date_message" not in attrs:
            continue

        neo4j_dt = attrs['date_message']
        python_dt = neo4j_dt.to_native() 
        timestamp = pd.to_datetime(python_dt)
        if pd.isna(timestamp):
            continue

        # Group ID
        group_id = attrs.get("id_group_anonymous")

        # Try to find the user who sent it:
        user_id = None

        # 1. Check predecessors (User → Message)
        preds = list(G.predecessors(node_id))
        for p in preds:
            p_attrs = G.nodes[p]
            if "User" in p_attrs.get("labels", []):
                user_id = p
                break

        # 2. Backup to message property (if exists)
        if user_id is None:
            user_id = attrs.get("id_member_anonymous") or attrs.get("id_member_anonymous")

        if user_id is None:
            continue 

        # Final row
        rows.append({
            "message_node": node_id,
            "user_id": user_id,
            "id_group_anonymous": group_id,
            "timestamp": timestamp,
        })

    return pd.DataFrame(rows)


In [3]:
def analyze_flood_attacks(G_rapid_shares):
    
    rapid_degree = dict(G_rapid_shares.degree())
    
    rapid_components = list(nx.connected_components(G_rapid_shares))
    botnet_suspects = set()
    for comp in rapid_components:
        if len(comp) > 2: # groups of 3+ people syncing perfectly
            botnet_suspects.update(comp)

        elif len(comp) == 2:
            user_a, user_b = list(comp)
            
            edge_data = G_rapid_shares.get_edge_data(user_a, user_b)
            sync_count = edge_data.get('weight', 1) 
            
            # If synced > 3 times, suspect.
            if sync_count >= 3:
                botnet_suspects.update(comp)

    all_users = set(G_rapid_shares.nodes())
    
    results = []
    
    for user in all_users:
        d_rapid = rapid_degree.get(user, 0)
        
        is_botnet_suspect = user in botnet_suspects
        
        results.append({
            "User": user,
            "Rapid_Sync_Count": d_rapid,
            "Botnet_Cluster": is_botnet_suspect
        })

    df = pd.DataFrame(results)
    
    return df

In [4]:
def find_top_flooding_users(df_messages, suspected_bots, window_seconds=10):
    bot_set = set(suspected_bots)

    df = df_messages.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp")

    results = []
    df["msg"] = 1 # dummy column for counting

    for user, user_df in df.groupby("user_id"):
        user_df = user_df.set_index("timestamp")

        rolling_count = user_df['msg'].rolling(
            f"{window_seconds}s",
            closed="both",
            min_periods=1
        ).count()

        max_burst = rolling_count.max()

        if max_burst > 1:
            results.append({
                "user_id": user,
                "max_messages_in_window": int(max_burst),
                "is_suspected_bot": user in bot_set
            })

    return pd.DataFrame(results).sort_values(
        "max_messages_in_window",
        ascending=False
    )

In [5]:
def find_top_flooded_groups(df_messages, suspected_bots, window_seconds=10):
    df = df_messages.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp")

    # Filter only messages sent by suspected bots
    df = df[df["user_id"].isin(suspected_bots)]

    results = []
    df["msg"] = 1 # dummy columns

    for group, gdf in df.groupby("id_group_anonymous"):
        gdf = gdf.set_index("timestamp")

        rolling_count = gdf['msg'].rolling(
            f"{window_seconds}s",
            closed="both",
            min_periods=1
        ).count()

        max_burst = rolling_count.max()

        results.append({
            "group_id": group,
            "max_messages_in_window": int(max_burst)
        })

    return pd.DataFrame(results).sort_values(
        "max_messages_in_window",
        ascending=False
    )

In [6]:
import traceback

try:
    driver = get_driver()

    print("Connecting to Neo4j and building NetworkX graph...")

    networkx_graph = load_msg_graph(driver)

    G_shares, G_viral, G_misinfo, G_rapid_shares = load_user_graphs(driver)

    df_messages = extract_messages_from_graph(networkx_graph)

    df_botinfo = analyze_flood_attacks(G_rapid_shares)
    suspected_bots = df_botinfo[df_botinfo["Botnet_Cluster"]]["User"].tolist()

    top_users = find_top_flooding_users(df_messages, suspected_bots)
    top_groups = find_top_flooded_groups(df_messages, suspected_bots)


    print("\n=== TOP FLOODERS ===")
    print(top_users.head(20))

    print("\n=== TOP FLOODERS (SUSPECTED BOTS) ===")
    bot_flooders = top_users[top_users["is_suspected_bot"] == True]
    print(bot_flooders.head(20))
    
    print("\n=== TOP FLOODED GROUPS ===")
    print(top_groups.head(20))

except Exception:
    traceback.print_exc()
finally:
    if 'driver' in locals() and driver:
        driver.close()
        print("Connection closed.")

Connecting to Neo4j and building NetworkX graph...


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=3, column=20, offset=42>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 42, 'line': 3, 'column': 20}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            MATCH (n)\n            RETURN id(n) AS id, labels(n) AS labels, properties(n) AS properties\n        '
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider


=== TOP FLOODERS ===
     user_id  max_messages_in_window  is_suspected_bot
226    24573                      29             False
36     21993                      28              True
891    49026                      23             False
163    23360                      18             False
490    28591                      17             False
254    24975                      17             False
143    23103                      17             False
261    25064                      17             False
869    47808                      16             False
168    23465                      14             False
874    48191                      14             False
68     22217                      13             False
47     22049                      13              True
819    44075                      12             False
142    23046                      12             False
296    25588                      12             False
773    41660                      11       